# FinSight-RAG — Phase 3 (Final System)
**A Source-Grounded Question-Answering Assistant for Corporate Financial Filings**
Lab 9 PSIS · Activity 1 (RAG-Based Domain Assistant) · Akash Goyal (I025) · Karan Lokhande (I039) · Meer Magia (I040)

**Before you start:** add your Gemini key in Colab: click the 🔑 **Secrets** icon in the left sidebar → *Add new secret* →
name `GOOGLE_API_KEY`, value = your key from Google AI Studio → turn **Notebook access** on. Then **Runtime → Run all**.
The key is read from Colab Secrets and is never printed or saved to the repository.

What this notebook does:
1. Clones the repo, installs the pinned packages and downloads the NIKE FY2023 10-K (107 pages).
2. Runs the full Phase 3 evaluation: semantic vs keyword vs **hybrid** retrieval, answers from Gemini for Q1–Q7
   (Phase 2 baseline vs Phase 3 final system), automatic checks, and the multi-turn follow-up test.
3. Prints every answer next to the full text of the passage it cites, for the manual grounding review.
4. Live demo cells (used in the video) and the test suite.

## 1. Setup

In [ ]:
%cd /content
!test -d finsight-rag-I025-39-40 || git clone -q https://github.com/akashgoyalll/finsight-rag-I025-39-40.git
%cd /content/finsight-rag-I025-39-40
!git pull -q
# If the Phase 3 code is not on GitHub yet: drag finsight-rag-phase3.zip into the Files panel first
# (any folder under /content; a renamed download such as "finsight-rag-phase3 (1).zip" is fine).
import glob, os, subprocess
zips = sorted(glob.glob("/content/**/finsight-rag-phase3*.zip", recursive=True), key=os.path.getmtime)
if zips:
    subprocess.run(["unzip", "-o", "-q", zips[-1], "-d", "."], check=True)
    print("Applied Phase 3 files from", zips[-1])
!git log --oneline -3
if not os.path.exists("scripts/run_phase3.py"):
    raise SystemExit("STOP: Phase 3 code missing. Push it to GitHub, or upload finsight-rag-phase3.zip in the Files panel, then re-run this cell.")
print("Phase 3 code present")
!pip install -q -r requirements.txt
!bash scripts/get_data.sh
!ls -la data

## 2. Gemini API key and model

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API key: ")
# Free-tier Gemini model (Google AI Studio). If section 3 stops with "quota exhausted" and you don't want
# to wait for the daily reset, switch to another free model, e.g. "google_genai:gemini-3.5-flash-lite",
# and re-run from here.
os.environ["LLM_MODEL"] = "google_genai:gemini-3.5-flash"
print("Key loaded:", bool(os.environ.get("GOOGLE_API_KEY")), "| model:", os.environ["LLM_MODEL"])

## 3. Full Phase 3 evaluation
Retrieval comparison for Q1–Q6 (semantic vs keyword vs **hybrid**, no LLM), then Gemini answers for Q1–Q7 with the final
system (hybrid retrieval, top-6), then the two-turn memory test with and without follow-up rewriting.

**Free-tier friendly:** about 12 Gemini calls, one every 13 seconds (under 5 requests/minute), so it takes about 5 minutes.
On a rate-limit error it waits the time Google asks for and retries. Results are saved after every answer.
**If it stops with "quota exhausted", just re-run this cell later (or after switching model in section 2)**: answers already
saved are reused, so no quota is wasted. Results: `outputs/phase3_results.json`, `outputs/phase3_summary.md`.

In [ ]:
import os, subprocess, sys
# Runs the evaluation, streams its output here and saves it to outputs/phase3_log_colab.txt
with open("outputs/phase3_log_colab.txt", "w") as log:
    p = subprocess.Popen([sys.executable, "scripts/run_phase3.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "unauthenticated requests" in line or "it/s]" in line:
            continue
        print(line, end=""); log.write(line)
    p.wait()
if p.returncode != 0 or not os.path.exists("outputs/phase3_summary.md"):
    raise SystemExit(f"STOP: run_phase3.py failed (exit code {p.returncode}). The error is in the last lines printed above.")
import json
if not json.load(open("outputs/phase3_results.json")).get("complete"):
    print("
INCOMPLETE (Gemini quota). Re-run this cell later; saved answers are reused.")

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("outputs/phase3_summary.md").read()))

## 4. Manual grounding review
For each answer of the final system: the answer, then the **full text of every chunk it cites** (document, page).
Grounded = every figure/claim in the answer appears in (or is directly computed from) the cited text.

In [ ]:
import json, sys, textwrap
sys.path.insert(0, "src")
from finsight.indexing import load_pages, chunk
from finsight.hybrid import chunk_key
chunks = chunk(load_pages(["data/nke-10k-2023.pdf"]))
by_key = {chunk_key(c): c for c in chunks}
res = json.load(open("outputs/phase3_results.json"))
final = [k for k in res["answers"] if k.startswith("final")][0]
for r in res["answers"][final]:
    print("=" * 100)
    print(f"{r['id']}  {r['question']}\nTRUTH : {r['ground_truth']}\nANSWER: {r['answer'] or r.get('error')}")
    print(f"answerable={r['answerable']}  auto answer_ok={r['answer_ok']}  cited_ok={r['cited_ok']}")
    for s in r["sources"]:
        c = by_key.get((s["source"], s["page"], s["start_index"], s["passage"][:50]))
        print(f"  --- cited [{s['excerpt']}] {s['source']} p.{s['page']} ---")
        print(textwrap.indent(textwrap.fill((c.page_content if c else s["passage"]), 110), "    "))

## 5. Live demo (used in the video)
Builds the final system in memory (hybrid retrieval + Gemini + memory) and answers questions with page citations.
Self-contained: you can skip sections 3–4 (no need to re-run the evaluation). Each question uses 1–2 Gemini calls;
on the free tier run these cells about 15 seconds apart.

In [ ]:
import warnings
from langchain_core._api import LangChainDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)
from finsight.indexing import index_chunks
from finsight.hybrid import build_hybrid_retriever
from finsight.chain import build_chain, with_memory
from finsight.llm import get_llm
from finsight.indexing import load_pages, chunk

# Uses Gemini (about 7 requests for the cells below; free tier: 5/minute, 20/day per model).
# Daily quota used up? Either use a key from a different Google Cloud project (quota is per project),
# or another free model, e.g.:  os.environ["LLM_MODEL"] = "google_genai:gemini-3.6-flash"
chunks = chunk(load_pages(["data/nke-10k-2023.pdf"]))
retriever = build_hybrid_retriever(index_chunks(chunks, kind="hf"), index_chunks(chunks, kind="tfidf"), fetch_k=4, k=6)
chat = with_memory(build_chain(retriever, get_llm()))

def ask(question, session="demo"):
    out = chat.invoke({"question": question}, config={"configurable": {"session_id": session}})
    print(f"Q: {question}")
    if out["retrieval_query"] != question:
        print(f"   (rewritten for retrieval: {out['retrieval_query']})")
    print(f"A: {out['answer']}\n   answerable = {out['answerable']}")
    for s in out["sources"]:
        print(f"   source [{s['excerpt']}]: {s['source']}, page {s['page']}  |  {s['passage'][:120].replace(chr(10), ' ')}...")
    print(f"   retrieved pages: {out['retrieved_pages']}\n")
    return out

In [ ]:
_ = ask("How many employees did NIKE have as of May 31, 2023?", session="demo-1")

In [ ]:
_ = ask("What were NIKE, Inc.'s total revenues in fiscal 2023?", session="demo-2")
_ = ask("How much did that grow on a currency-neutral basis?", session="demo-2")   # follow-up uses memory

In [ ]:
_ = ask("Which NIKE Brand geographic segment had the highest EBIT margin in fiscal 2023?", session="demo-3")  # weak case

In [ ]:
_ = ask("What were NIKE's total revenues in fiscal 2025?", session="demo-4")  # not in the filing -> should refuse

In [ ]:
# Try your own question
_ = ask("What was NIKE's diluted earnings per share in fiscal 2023?", session="demo-5")

## 6. Test suite (offline, no API calls)

In [ ]:
!python -m pytest tests/ -v -p no:cacheprovider -W ignore::DeprecationWarning 2>&1 | tee outputs/phase3_pytest_log_colab.txt | tail -16

## 7. Download the results
Downloads `phase3_outputs.zip` (results JSON, summary, logs). Send it to the report owner / commit it to `outputs/`.

In [ ]:
!cd outputs && zip -q -j ../phase3_outputs.zip phase3_results.json phase3_summary.md phase3_log_colab.txt phase3_pytest_log_colab.txt
from google.colab import files
files.download("phase3_outputs.zip")